# Notebook 3: Tumor Analysis

This notebook focuses on tumor cell subclustering, pathway activity scoring, genomic concordance analysis, and tumor versus normal epithelium comparison.

**Overview:**
1. Tumor subclustering with GPU acceleration (rapids_singlecell)
2. Per-cell pathway scoring via ssGSEA/enrichr (gseapy, MSigDB Hallmark 2020)
3. Genomic concordance: KRAS, PIK3CA, MMR status from exome data
4. Tumor vs. normal epithelium comparison
5. Tumor subcluster composition across response groups and timepoints

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from multiprocessing import Pool
import gseapy as gp
import scanpy as sc
import rapids_singlecell as rsc
import anndata

sc.settings.verbosity = 3
sc.settings.figdir = '../figures/'
plt.rcParams['figure.dpi'] = 150

## 2. Load Tumor Data

In [ ]:
# Load tumor cell subset
adata_tumor = sc.read_h5ad('../data/integrate_adata_filtered_tumor.h5ad')
print('Tumor cells:', adata_tumor)

# Load tumor + normal combined object for comparison
adata_tumor_normal = sc.read_h5ad('../data/integrate_adata_filtered_tumor_normal.h5ad')
print('Tumor + Normal:', adata_tumor_normal)

## 3. Tumor Subclustering (GPU-Accelerated)

Tumor cells are clustered using rapids_singlecell for GPU-accelerated PCA, UMAP, and Leiden clustering.

In [ ]:
# GPU-accelerated clustering pipeline
rsc.get.anndata_to_GPU(adata_tumor)
rsc.tl.pca(adata_tumor)
rsc.pp.neighbors(adata_tumor, random_state=0)
rsc.tl.umap(adata_tumor, random_state=0)
rsc.tl.leiden(adata_tumor, resolution=0.5, random_state=0)
rsc.get.anndata_to_CPU(adata_tumor)

# Prefix leiden labels with 'Tumor '
adata_tumor.obs['leiden'] = ['Tumor c' + str(i) for i in adata_tumor.obs['leiden']]
print('Tumor leiden clusters:', sorted(adata_tumor.obs['leiden'].unique()))

adata_tumor.write_h5ad('../data/integrate_adata_filtered_tumor_clustered.h5ad')

In [ ]:
# Differential expression for tumor subclusters
sc.tl.rank_genes_groups(adata_tumor, groupby='leiden', method='wilcoxon', pts=True)
deg_tumor = sc.get.rank_genes_groups_df(adata_tumor, group=None)
deg_tumor.to_csv('../results/deg_tumor_subclusters.csv', index=False)

# Visualize top markers
sc.pl.rank_genes_groups_dotplot(
    adata_tumor, n_genes=5, groupby='leiden',
    show=False
)
plt.savefig('../figures/03_dotplot_tumor_markers.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Per-Cell Pathway Scoring (ssGSEA via Enrichr)

Pathway activity is estimated per cell using Enrichr with the MSigDB Hallmark 2020 gene set collection. Expressed genes (count > 0) are used as the input gene list for each cell.

In [ ]:
# Extract expression matrix as DataFrame
df_exp = adata_tumor.to_df()

def run_enrichr_cell(i_row):
    """Run Enrichr pathway scoring for a single cell.
    
    Uses genes with non-zero expression as the input gene list.
    Returns a DataFrame with enrichment results for the cell.
    """
    i, exp_cell = i_row
    deg_ls = exp_cell[exp_cell != 0].index.tolist()
    if len(deg_ls) < 5:
        return None
    try:
        enr = gp.enrichr(
            gene_list=deg_ls,
            gene_sets='MSigDB_Hallmark_2020.gmt',
            organism='human',
            outdir=None,
        )
        df_temp = enr.results.copy()
        df_temp['cell'] = i
        return df_temp
    except Exception:
        return None

In [ ]:
# Run enrichr in parallel (adjust n_workers based on available CPU cores)
n_workers = 8
with Pool(n_workers) as pool:
    results = pool.map(run_enrichr_cell, list(df_exp.iterrows()))

# Concatenate valid results
df_enrichr = pd.concat([r for r in results if r is not None], ignore_index=True)
df_enrichr.to_csv('../results/tumor_enrichr_hallmark.csv', index=False)
print('Enrichr results shape:', df_enrichr.shape)

In [ ]:
# Pathways of interest for tumor biology
pathways_of_interest = [
    'Hypoxia',
    'DNA Repair',
    'Epithelial Mesenchymal Transition',
    'Angiogenesis',
    'WNT Beta Catenin Signaling',
    'KRAS Signaling Up',
    'PI3K AKT MTOR Signaling',
    'Glycolysis',
    'TGF Beta Signaling',
    'P53 Pathway',
]

# Pivot: cells × pathways, using -log10(adjusted p-value) as score
df_pivot = (
    df_enrichr[df_enrichr['Term'].isin(pathways_of_interest)]
    .pivot_table(index='cell', columns='Term', values='Adjusted P-value', aggfunc='min')
    .fillna(1.0)
    .apply(lambda x: -np.log10(x + 1e-10))
)

# Add pathway scores to adata obs
df_pivot.columns = [f'pathway_{c.replace(" ", "_")}' for c in df_pivot.columns]
for col in df_pivot.columns:
    adata_tumor.obs[col] = df_pivot[col].reindex(adata_tumor.obs_names).values

print('Pathway scores added to adata_tumor.obs')

In [ ]:
# Visualize pathway scores on UMAP
pathway_cols = [c for c in adata_tumor.obs.columns if c.startswith('pathway_')]

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
for ax, col in zip(axes.flat, pathway_cols[:10]):
    sc.pl.umap(adata_tumor, color=col, ax=ax, show=False,
               title=col.replace('pathway_', '').replace('_', ' '),
               color_map='RdYlBu_r')
plt.suptitle('Hallmark Pathway Activity in Tumor Cells', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../figures/03_umap_tumor_pathways.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Genomic Concordance Analysis

Correlates single-cell pathway activity with patient-level genomic alterations (KRAS mutation, PIK3CA mutation, MMR status) from exome sequencing data.

In [ ]:
# Load exome data (patient-level genomic alterations)
df_exome = pd.read_csv('../data/exome.txt', sep='\t', index_col=0)
print('Exome data columns:', df_exome.columns.tolist())
print(df_exome.head())

In [ ]:
# Annotate tumor cells with patient genomic status
def get_patient_id(sample_name):
    return sample_name.split('_')[0]  # e.g., 'Pt-1_Pre' -> 'Pt-1'

adata_tumor.obs['Patient'] = adata_tumor.obs['Sample'].apply(get_patient_id)

# Map genomic alterations to cells
for col in ['RAS', 'PIK3CA', 'MMR']:
    if col in df_exome.columns:
        adata_tumor.obs[col] = adata_tumor.obs['Patient'].map(df_exome[col])

print('Genomic annotations added:', [c for c in ['RAS', 'PIK3CA', 'MMR'] if c in adata_tumor.obs.columns])

In [ ]:
# Compare KRAS signaling pathway score between KRAS mutant and wild-type tumors
if 'RAS' in adata_tumor.obs.columns and 'pathway_KRAS_Signaling_Up' in adata_tumor.obs.columns:
    kras_mut = adata_tumor.obs.loc[
        adata_tumor.obs['RAS'] == 'Mut', 'pathway_KRAS_Signaling_Up'
    ].dropna()
    kras_wt = adata_tumor.obs.loc[
        adata_tumor.obs['RAS'] == 'WT', 'pathway_KRAS_Signaling_Up'
    ].dropna()

    stat, pval = stats.mannwhitneyu(kras_mut, kras_wt, alternative='two-sided')
    print(f'KRAS Mut vs WT - KRAS Signaling pathway: U={stat:.0f}, p={pval:.4f}')

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.boxplot(
        data=adata_tumor.obs.dropna(subset=['RAS', 'pathway_KRAS_Signaling_Up']),
        x='RAS', y='pathway_KRAS_Signaling_Up',
        palette={'Mut': '#e41a1c', 'WT': '#377eb8'}, ax=ax
    )
    ax.set_title(f'KRAS Signaling Activity\nMut vs WT (p={pval:.4f})')
    ax.set_ylabel('-log10(adj. p-value)')
    plt.tight_layout()
    plt.savefig('../figures/03_kras_pathway_boxplot.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Tumor vs. Normal Epithelium Comparison

In [ ]:
# DEG: Tumor vs Normal epithelium
sc.tl.rank_genes_groups(
    adata_tumor_normal,
    groupby='Major_cluster_pathol',
    groups=['Tumor'],
    reference='Normal',
    method='wilcoxon',
    pts=True
)

deg_tumor_vs_normal = sc.get.rank_genes_groups_df(adata_tumor_normal, group='Tumor')
deg_tumor_vs_normal.to_csv('../results/deg_tumor_vs_normal.csv', index=False)
print('Top tumor-enriched genes:')
print(deg_tumor_vs_normal.head(20)[['names', 'scores', 'pvals_adj', 'logfoldchanges']])

In [ ]:
# Volcano plot: Tumor vs Normal
deg = deg_tumor_vs_normal.copy()
deg['-log10_pval'] = -np.log10(deg['pvals_adj'].clip(lower=1e-300))
deg['significant'] = (deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'].abs() > 1)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    deg.loc[~deg['significant'], 'logfoldchanges'],
    deg.loc[~deg['significant'], '-log10_pval'],
    s=2, alpha=0.3, color='grey'
)
ax.scatter(
    deg.loc[deg['significant'], 'logfoldchanges'],
    deg.loc[deg['significant'], '-log10_pval'],
    s=4, alpha=0.6, color='red'
)
# Label top genes
top_genes = deg[deg['significant']].nlargest(15, '-log10_pval')
for _, row in top_genes.iterrows():
    ax.annotate(row['names'], (row['logfoldchanges'], row['-log10_pval']),
                fontsize=7, alpha=0.8)
ax.axvline(x=1, color='black', linestyle='--', linewidth=0.8)
ax.axvline(x=-1, color='black', linestyle='--', linewidth=0.8)
ax.axhline(y=-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Log2 Fold Change (Tumor vs Normal)')
ax.set_ylabel('-log10(adj. p-value)')
ax.set_title('Tumor vs. Normal Epithelium DEGs')
plt.tight_layout()
plt.savefig('../figures/03_volcano_tumor_vs_normal.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Tumor Subcluster Composition by Response Group

In [ ]:
# Annotate response group
def get_response(sample_name):
    pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
    if pt_num <= 8:
        return 'SD'
    elif pt_num <= 16:
        return 'MPR'
    else:
        return 'CR'

adata_tumor.obs['Response'] = adata_tumor.obs['Sample'].apply(get_response)
adata_tumor.obs['Timepoint'] = adata_tumor.obs['Sample'].apply(lambda s: s.split('_', 1)[1])

# Composition barplot: tumor subclusters per sample
composition = (
    adata_tumor.obs.groupby(['Sample', 'leiden'])
    .size()
    .unstack(fill_value=0)
)
composition = composition.div(composition.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(20, 5))
composition.plot(kind='bar', stacked=True, ax=ax, edgecolor='none', width=0.9)
ax.set_title('Tumor Subcluster Composition per Sample')
ax.set_xlabel('Sample')
ax.set_ylabel('Cell proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.savefig('../figures/03_composition_tumor_subclusters.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
adata_tumor.write_h5ad('../data/integrate_adata_filtered_tumor_annotated.h5ad')
print('Saved integrate_adata_filtered_tumor_annotated.h5ad')